In [ ]:
source("~/heads/head.r")

In [ ]:
### meta.data ###

m1 = read_xlsx("./Sample_summary.xlsx",sheet = 1)
m1 = m1 %>% filter(!Sample %in% paste0("SC_RCC_",c("08","17","22","23","25","34","63","77")))
l = readRDS("../2025/scrcc/data/sample_subject_list.rds")

m2 = read_xlsx("./Sample_summary.xlsx",sheet = 2) %>% as.data.frame
m2 = m2 %>% filter(!Sample %in% c("8","17","22","23","25","34","63","77"))
m2 = m2 %>% select(`scRNAseq ID`,Batch) %>% setnames(new = c("Sample","lab_batch")) # %>% left_join(m2,by="Sample")

# batch = read_xlsx("./Sample_summary.xlsx",sheet = 4)

In [ ]:
# sample-level

m2$sample_id1 = m2$Sample
m2$sample_id2 = gsub("_","",m2$Sample)
m2$sample_id3 = gsub("SC_RCC_","SC_",m2$Sample)

m2$subject_id = l$subjects[match(m2$sample_id2,names(l$subjects))] # subject_id used in tcr/bcr clone calling
m2$subject_id[is.na(m2$subject_id)] = gsub("NORM|NK","",m2$sample_id2)[is.na(m2$subject_id)]

m2$tissue = ifelse(grepl("NORM|NK",m2$sample_id1),"normal","tumor")

# m2$in_scrna = m2$sample_id2 %in% paste0("SCRCC",c("08","17","22","23","25","34","63","77"))
m2$in_sctcr = !m2$sample_id2 %in% paste0("SCRCC",c("14NORM","15")) # sequencing failed, 14NORM and 15 missing cellranger outputs & deprecated fastq files
m2$in_scbcr = !m2$sample_id2 %in% paste0("SCRCC",c("49","77")) # sequencing failed, 49 low bcr concentration
m2$in_sccite = m2$sample_id2 %in% paste0("SCRCC",c("03","08","10","29","58","59")) # 08 was removed from scrna data though

# m2$in_bulkrna = 
# m2$in_xenium = 
m2$in_codex = m2$sample_id2 %in% paste0("SCRCC",c("03","29","58","59")) 

# Batch
lb = m2$lab_batch
m2$lab_batch = NULL
m2$lab_batch = lb

b = fread("./SC_RCC_updated_annotations_with_fastq(SC_RCC_updated_annotations_2023).csv")
b = b %>% filter(!Sample %in% c("8","17","22","23","25","34","63","77"))
b$`scRNAseq ID`[b$`scRNAseq ID` == "SC_RCC_21_NK"] = "SC_RCC_21_NORM"

# b$seq_batch_rna = basename(b$`RNA fastq location`)
# b$seq_batch_tcr = basename(b$`TCR fastq location`)
# b$seq_batch_bcr = basename(b$`BCR fastq location`)

b$seq_batch_rna = gsub("^.*MEDCCC/","",gsub("YCGA.*$","YCGA",b$`RNA fastq location`))
b$seq_batch_tcr = gsub("^.*MEDCCC/","",gsub("YCGA.*$","YCGA",b$`TCR fastq location`))
b$seq_batch_bcr = gsub("^.*MEDCCC/","",gsub("YCGA.*$","YCGA",b$`BCR fastq location`))
b$seq_batch_bcr[b$Sample == "49"] = "missing"

m2 = cbind(m2,b[match(m2$Sample,b$`scRNAseq ID`),c("seq_batch_rna","seq_batch_tcr","seq_batch_bcr")])

In [ ]:
# sample mapping
cite = data.frame(
    scrna_id = c(paste0("SCRCC",c("03","08","10","29","58","59")), NA),
    sccite_id = c(809529, 944137, 940798, 853378, 847960, 810760, 930847), # no tumor available for 930847
    codex_id = c("RCC1",NA,NA,"RCC2","RCC4","RCC3",NA)
)
cite

In [ ]:
# patient-level

m1$primary_histology = case_when(m1$Cancer_type== "ccRCC"~"ccRCC",
                                m1$Cancer_type %in% c("chromophobe","Chromophobe")~"chRCC",
                                m1$Cancer_type == "Papillary"~"ccRCC",
                                m1$Cancer_type == "Renal Medullary carcinoma"~"RMC",
                                m1$Cancer_type == "EVT RCC"~"EVT",
                                m1$Cancer_type %in% c("unclassified","Unclassified","Unclassified / Undifferentiated")~"uRCC",
                                m1$Sample %in% c("SC_RCC_78","SC_RCC_79")~"Oncocytoma",
                                m1$Sample == "SC_RCC_69"~"LungCancer")

m1$sarcomatoid_rhabdoid = case_when(m1$Sarcomatoid_Rhabdoid == "NA" | is.na(m1$Sarcomatoid_Rhabdoid) ~ "missing",
                                    m1$Sarcomatoid_Rhabdoid == 0 ~ "yes",
                                    m1$Sarcomatoid_Rhabdoid == 1 ~ "no")

m1$treatment = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    TRUE~gsub(" ", "", m1$Regent))
m1$treatment_type = ifelse(
    grepl("ICI",m1$treatment),
    "ICI_based",
    m1$treatment)

m1$treatment_timepoint = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    m1$treatment == "NoTx" ~ "napply",
    TRUE~m1$Timepoint)
m1$treatment_line = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    m1$treatment == "NoTx" ~ "napply",
    #m1$First_line == 1 ~ 1,
    TRUE~gsub("..\\)$","",gsub("0 \\(","",m1$First_line)))

m1$response_best = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    m1$treatment == "NoTx" ~ "napply",
    TRUE~gsub(" ","",m1$Best_response))
m1$response_collecting = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    m1$treatment == "NoTx" ~ "napply",
    m1$treatment_timepoint == "Pre" ~ "napply",
    TRUE~gsub(" ","",m1$Response_at_collecting_time))

CB = paste0('SC_RCC_',c('24', '05', '10', '50', '38', '42', '80', '37', '03', '54', '64', '12', '29', '09', '21', '53', '59', '16'))
NCB = paste0('SC_RCC_',c("01", "02", "47", "45", "46", "55", "06", "40", "74", "11", "07"))
m1$response = case_when(m1$Sample %in% CB~"R",
                        m1$Sample %in% NCB~"NR",
                        TRUE~NA) # only for ICI-based therapy

m1$PFS[m1$Regent == "Chemo"] = "NA" # correction

m1$PFS_time = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    m1$treatment == "NoTx" ~ "napply",
    m1$Best_response == "NE" ~ "NE",
    m1$treatment_type != "ICI_based" ~ "missing", # not inferred by Soki yet
    TRUE~paste0(m1$PFS,"d"))
m1$PFS_status = case_when(
    !m1$PFS_time %in% c("napply","missing","NE") ~ "1", # 1-progression happened, need to confirm with Soki
    TRUE ~ m1$PFS_time)

m1$tumor_site = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    TRUE~m1$Tumor_site)

m1$on_steroids = case_when(
    m1$Sample %in% c("SC_RCC_69","SC_RCC_78","SC_RCC_79") ~ "missing",
    m1$On_steroids=="n" ~ "no",
    m1$On_steroids=="y" ~ "yes")

In [ ]:
m1 = m1 %>% select(Sample,primary_histology,sarcomatoid_rhabdoid,treatment,treatment_type,treatment_timepoint,treatment_line,response_best,response_collecting,response,PFS_time,tumor_site,on_steroids)

m_t = left_join(m2 %>% filter(tissue=="tumor"), m1, by="Sample")
m_n = m2 %>% filter(tissue=="normal") %>% mutate(Sample=gsub("_(NORM|NK)","",Sample)) %>% left_join(m1, by="Sample")

m = rbind(m_t,m_n) %>% select(-Sample) %>% arrange(sample_id1)
write.table(m,file = "./scrcc_metadata.csv",quote = F,sep = ",",row.names = F)

In [ ]:
###########################################################################################################